# 05 · Injury labels + linking to performance data

Project 1 (Injury Risk Modelling). This shows the **curated injury dataset**
(labels + identity mapping in one file) and how it links to club + public data.
Requires your read-only key in `.env` (see `docs/data_access.md`).


In [ ]:
import nffc_data as nffc
from nffc_data import external


## 1. Injury labels + identity mapping (from the bucket)
One row per injury spell; the id columns are the bridge to the other datasets.


In [ ]:
inj = nffc.load_parquet("injuries/gb1_injuries_with_mapping.parquet")
print(inj.shape)
inj[["player_name","statsbomb_id","second_spectrum_id","team_name","season","reason","from","until","days_missed"]].head()


### Injury reasons (needs normalising — free text)


In [ ]:
inj["reason"].value_counts().head(10)


## 2. Per-player injury burden
Aggregate spells into player-season features (days/games missed, spell count).


In [ ]:
burden = (inj.groupby(["player_name","season"])
            .agg(spells=("reason","size"), days_missed=("days_missed","sum"),
                 games_missed=("games_missed","sum"))
            .reset_index().sort_values("days_missed", ascending=False))
burden.head()


## 3. Weekly availability proxy from the FPL archive (public)
FPL `minutes` per gameweek → a run of 0-minute GWs after playing ≈ unavailable.
(See docs/external_data.md for the caveats.)


In [ ]:
gw = external.load_fpl_gameweeks("2023-24")
print(gw.shape)
gw[["name","GW","minutes","starts","kickoff_time"]].head()


## 4. Linking injuries to club data
The mapping is embedded in the injury file:
- `statsbomb_id` → StatsBomb events/lineups `player_id`
- `second_spectrum_id` → SecondSpectrum tracking player id

StatsBomb / tracking / GPS are uploaded as access is rolled out; once present:


In [ ]:
# matches = nffc.load_parquet("Statsbomb/Premier League/2023-2024/matches.parquet")
# events  = nffc.load_parquet("Statsbomb/Premier League/2023-2024/events/<match_id>.parquet")
# joined  = events.merge(inj, left_on="player_id", right_on="statsbomb_id", how="left")
print("injury players with a StatsBomb id:", inj["statsbomb_id"].notna().sum())
print("injury players with a SecondSpectrum id:", inj["second_spectrum_id"].notna().sum())


## 5. Optional: public Transfermarkt injuries for non-curated players
`player_id` here is the Transfermarkt id, so you can pull the full public
Transfermarkt injury history and profiles the same way.


In [ ]:
# tm_inj  = external.load_transfermarkt_injuries()        # salimt player_injuries.csv
# tm_prof = external.load_transfermarkt_profiles()        # tm_id -> name/DOB/position
print("see docs/external_data.md")
